# Module 8.2: Scaling Laws

Why do *bigger* models, *more* data, and *more* compute make LLMs better — and given a fixed compute budget, how do we spend it optimally?

## 1. The intuition: why scale works at all

You've now built a complete, efficient Transformer (Parts 3-4, 7-8). A natural question follows: **if I just make it bigger, does it get smarter?** Astonishingly, the answer is *yes* — and in a way so regular you can graph it.

Three knobs drive a model's quality:

- **Parameters ($N$)** — capacity. More weights means more room to *store patterns*: spellings, grammar, facts, idioms. A tiny model literally cannot hold enough structure to be fluent, the way a one-page notebook can't hold an encyclopedia.
- **Data ($D$)** — experience. More tokens means more of the world to *learn from*. A huge model trained on three sentences just memorizes those sentences.
- **Compute ($C$)** — practice. More FLOPs means more gradient steps to actually *fit* the parameters to the data. Capacity and experience are useless if you never finish studying.

The remarkable empirical discovery (Kaplan et al. 2020; Hoffmann et al. 2022, "Chinchilla") is that test **loss falls predictably** as you turn these knobs — not erratically, but along a smooth curve you can *extrapolate*. That predictability is what lets labs promise, before spending millions of dollars, roughly how good the next model will be.

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
print("PyTorch", torch.__version__)

## 2. The power law, intuitively

When researchers plotted test loss against scale, they didn't get a random scatter — they got a **straight line on a log-log plot**. A straight line in log-log space *is* a power law:

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha} + L_\infty$$

Don't sweat the symbols. The shape is the whole point:

- Loss keeps **dropping** as $N$ grows — bigger really is better.
- But with **diminishing returns**: each *doubling* of size buys a *constant* drop in loss, not a constant *fraction* of the remaining gap. Going from 1M to 10M params helps about as much as going from 100M to 1B.
- There's an irreducible floor $L_\infty$ — the entropy of language itself. You can't predict perfectly; some text is genuinely unpredictable.

Let's just *draw* a power law so the shape is in your eyes before we see it in a real model.

In [ ]:
# A toy power law: L(N) = (Nc / N)^alpha + L_inf
N = torch.logspace(6, 11, 100)   # model sizes from 1e6 to 1e11 params
Nc, alpha, L_inf = 1e8, 0.076, 1.7
L = (Nc / N) ** alpha + L_inf

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(N, L)
axes[0].set(title="Linear axes: curve bends, hard to read",
            xlabel="parameters N", ylabel="test loss L")

axes[1].plot(N, L)
axes[1].set_xscale("log")
axes[1].set(title="Log-log axes: a near-straight line = power law",
            xlabel="parameters N (log)", ylabel="test loss L")
axes[1].axhline(L_inf, ls="--", color="gray", label=f"irreducible floor L_inf={L_inf}")
axes[1].legend()
plt.tight_layout(); plt.show()
print("Notice: every 10x in N buys a similar-sized drop in L -> diminishing returns.")

The same straight-line story holds (with different exponents) for **data** $D$ and **compute** $C$. Three knobs, three power laws. The practical magic: measure a handful of small runs, fit the line, and *extrapolate* to predict a model you haven't built yet.

## 3. Compute-optimal allocation (the Chinchilla lesson)

Here's the catch that makes scaling laws *actionable*. Compute isn't free — you have a **budget** $C$ (GPU-hours, dollars). And a famous rule of thumb ties the three knobs together:

$$C \approx 6 \, N \, D$$

(roughly 6 FLOPs per parameter per token: ~2 for the forward pass, ~4 for the backward pass.)

This equation forces a **trade-off**. For a *fixed* $C$, if you spend it on a bigger model ($N$ up) you have fewer tokens to train on ($D$ down), and vice versa. So: do you train a **big model on little data**, or a **smaller model on lots of data**?

For years the field over-weighted $N$. **GPT-3** was enormous (175B params) but, in hindsight, *undertrained* — fed far too few tokens for its size. Hoffmann et al.'s **Chinchilla** (2022) re-ran the experiment carefully and found the optimum is to **scale $N$ and $D$ together** — roughly **~20 tokens per parameter**. Their 70B Chinchilla, trained on 4x more data, *beat* the 175B Gopher using the **same** compute.

Let's turn that into a calculator. Given a budget $C$, plug $D \approx 20N$ into $C = 6ND$ and solve for the compute-optimal $N$ (then $D$).

In [ ]:
def chinchilla_optimal(C, tokens_per_param=20.0):
    """Given a compute budget C (in FLOPs), return the compute-optimal
    model size N and dataset size D under C = 6*N*D and D = ratio*N."""
    # C = 6 * N * (ratio * N) = 6 * ratio * N^2  ->  N = sqrt(C / (6*ratio))
    N = math.sqrt(C / (6.0 * tokens_per_param))
    D = tokens_per_param * N
    return N, D

def human(x):
    for unit, size in [("T", 1e12), ("B", 1e9), ("M", 1e6), ("K", 1e3)]:
        if x >= size:
            return f"{x/size:5.1f}{unit}"
    return f"{x:.0f}"

print(f"{'compute budget C':>18} | {'optimal params N':>16} | {'optimal tokens D':>16}")
print("-" * 60)
for C in [1e18, 1e19, 1e21, 1e23, 6e23]:
    N, D = chinchilla_optimal(C)
    print(f"{C:>18.0e} | {human(N):>16} | {human(D):>16}")

# Sanity check: does our allocation actually spend the budget? (6*N*D should == C)
N, D = chinchilla_optimal(1e21)
print(f"\nCheck for C=1e21: 6*N*D = {6*N*D:.2e} (should match 1e21)")

Read the table as a recipe: *"I have this much compute, therefore build a model of about this size and feed it about this many tokens."* Notice both $N$ and $D$ grow like $\sqrt{C}$ — when your budget goes up 100x, you make the model ~10x bigger **and** train on ~10x more data. Pouring everything into size alone (the GPT-3 mistake) leaves performance on the table.

## 4. A baby scaling law (empirical mini-demo)

Time to *see* the power law emerge from real training. We'll train a few **tiny** char-level GPTs of increasing size on the **same** small embedded corpus for the **same** small number of steps, then plot final loss vs. parameter count on log-log axes.

> **Honesty disclaimer.** This is a noisy toy at miniature scale — kilobytes of text, a few hundred steps, models thousands of times smaller than the smallest "real" LM. The *exact* numbers will wobble. But the **characteristic shape** comes through: more parameters -> lower loss, with diminishing returns. That's the whole lesson, just shrunk to run in seconds.

In [ ]:
from llm_workout.model import GPT

# --- A small embedded corpus. No network needed. ---
corpus = (
    "the scaling laws of language models are surprisingly smooth. "
    "as we grow the number of parameters, the data, and the compute, "
    "the test loss falls along a clean power law. bigger models store "
    "more patterns; more data gives more to learn from; more compute "
    "means more steps of practice. the returns diminish, yet they keep "
    "coming, doubling after doubling, until the cost outruns the gain. "
    "chinchilla taught us to balance the model and the data together, "
    "spending each unit of compute where it helps the most.\n"
) * 6

chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)
print(f"corpus chars={len(corpus)}  vocab_size={vocab_size}  tokens={len(data)}")

block_size = 32
max_seq_len = 64  # >= block_size; we only do forward/loss, never long generation

def get_batch(batch_size=16):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

def train_tiny(d_model, num_layers, num_heads, steps=250, lr=3e-3):
    torch.manual_seed(0)  # same init seed so size is the only variable
    model = GPT(vocab_size=vocab_size, d_model=d_model, num_layers=num_layers,
                num_heads=num_heads, hidden_dim=4 * d_model, max_seq_len=max_seq_len)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        x, y = get_batch()
        _, loss, _ = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
    # Final loss: average a few batches to smooth noise
    model.eval()
    with torch.no_grad():
        final = sum(model(*get_batch())[1].item() for _ in range(20)) / 20
    return count_params(model), final

# Increasing model sizes (vary width and depth). Each is tiny -> fast.
# Vary ONLY width (d_model) at fixed depth, so capacity increases smoothly
# and the optimization stays stable -- isolating size as the single variable.
configs = [
    dict(d_model=16, num_layers=2, num_heads=2),
    dict(d_model=24, num_layers=2, num_heads=2),
    dict(d_model=32, num_layers=2, num_heads=2),
    dict(d_model=48, num_layers=2, num_heads=2),
    dict(d_model=64, num_layers=2, num_heads=2),
]

results = []
for cfg in configs:
    n_params, final_loss = train_tiny(**cfg)
    results.append((n_params, final_loss))
    print(f"d_model={cfg['d_model']:>3} layers={cfg['num_layers']}  "
          f"params={n_params:>7,}  final_loss={final_loss:.4f}")

In [ ]:
params = [r[0] for r in results]
losses = [r[1] for r in results]

plt.figure(figsize=(7, 5))
plt.plot(params, losses, "o-", color="crimson")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("parameter count N (log)")
plt.ylabel("final training loss (log)")
plt.title("Baby scaling law: more params -> lower loss")
plt.grid(True, which="both", ls=":", alpha=0.5)
for n, l in results:
    plt.annotate(f"{n:,}", (n, l), textcoords="offset points", xytext=(6, 6), fontsize=8)
plt.tight_layout(); plt.show()

print("Downward trend confirmed:" if losses[-1] < losses[0]
      else "Hmm, noisy run -- try more steps.")

There it is — even in this cartoon-sized experiment, growing the model **pushes the loss down**, and the points fall *roughly* along a line in log-log space. The drop also gets **shallower** for the largest model: diminishing returns, exactly as the theory promised. The real Kaplan/Chinchilla plots are this same picture, just spanning many orders of magnitude with far less noise.

## 5. Caveats: where the clean story bends

Scaling laws are powerful but not magic. Keep these in mind:

- **Power laws eventually bend.** Extrapolation works over a range, but no curve falls forever — the irreducible floor $L_\infty$ (the entropy of language) is real, and other regimes can kick in.
- **Data *quality*, not just quantity.** "More tokens" assumes *good* tokens. Deduplicated, filtered, high-quality data shifts the whole curve down; garbage data wastes compute. Recent models lean hard on curation.
- **The "emergent abilities" debate.** Some skills seem to appear *suddenly* at a certain scale. Whether these are true phase transitions or artifacts of harsh, all-or-nothing metrics is genuinely contested — measure with smoother metrics and many "emergences" look gradual.
- **Loss is not capability.** Scaling laws predict *test loss*, which correlates with — but is not identical to — being helpful, factual, or safe. A lower loss doesn't guarantee better reasoning on a specific downstream task. That's why we still *evaluate* (Module 5.5) and *align* (Part 6).

## 6. Why this whole course mattered

Scaling laws explain the *strategy* of modern LLMs: pick a compute budget, allocate $N$ and $D$ optimally, and ride the power law down. But that strategy is only **affordable** because of *efficiency*.

Every architectural trick you built was, secretly, a scaling-law enabler:
- **RoPE, RMSNorm, SwiGLU** (Parts 3-4) — more loss-reduction per FLOP.
- **KV caching, FlashAttention, paged/advanced attention** (Part 7) — the same compute stretches over more tokens, raising effective $D$.
- **Mixture of Experts** (Module 8.1) — grows $N$ (capacity) *without* paying full compute per token, bending the $C \approx 6ND$ relationship in your favor.

Efficiency is what makes scaling buyable. And once you've spent the budget and trained the model, the last question is: how do you actually *serve* it to users, fast and cheap? That's **Part 9: Production** — quantization, speculative decoding, and the serving stack. Onward.

### 🏋️ Try it yourself

Make the scaling law your own:

1. **Vary data, not size.** Hold the model fixed and instead train on truncated corpora of growing length (e.g. `data[:n]` for several `n`). Plot final loss vs. *tokens* on log-log axes — you should see a data scaling law.
2. **Fit the line.** Use `numpy.polyfit(np.log(params), np.log(losses), 1)` to estimate the power-law exponent (the slope) from your baby experiment.
3. **Budget planning.** Use `chinchilla_optimal` to find $N$ and $D$ for a budget you can actually afford on a laptop GPU, then check whether the implied token count even exists in a dataset you have.

A starter for #2 is below.

In [ ]:
import numpy as np

# TODO: estimate the power-law slope from your baby experiment.
# A steeper (more negative) slope means scale buys you more.
log_N = np.log(np.array(params))
log_L = np.log(np.array(losses))

slope, intercept = np.polyfit(log_N, log_L, 1)
print(f"fitted power-law slope (alpha-ish) = {slope:.4f}")
print("Negative slope -> loss decreases as params grow. Try changing 'steps' above")
print("and see how the slope and noise respond. Your turn!")